In [ ]:
# Step 12: Channel Flow with Navier–Stokes
# 第 12 步：纳维-斯托克斯方程下的通道流
# 向 u 动量方程中添加一个源项，以模拟压力驱动通道流的影响
	
import numpy
from matplotlib import pyplot, cm
from mpl_toolkits.mplot3d import Axes3D

# 定义压力方程右项b
def build_up_b(rho, dt, dx, dy, u, v):
    b = numpy.zeros_like(u)
    b[1:-1, 1:-1] = (rho * (1 / dt * ((u[1:-1, 2:] - u[1:-1, 0:-2]) / (2 * dx) +
                                      (v[2:, 1:-1] - v[0:-2, 1:-1]) / (2 * dy)) -
                            ((u[1:-1, 2:] - u[1:-1, 0:-2]) / (2 * dx))**2 -
                            2 * ((u[2:, 1:-1] - u[0:-2, 1:-1]) / (2 * dy) *
                                 (v[1:-1, 2:] - v[1:-1, 0:-2]) / (2 * dx))-
                            ((v[2:, 1:-1] - v[0:-2, 1:-1]) / (2 * dy))**2))
    # 周期性假设 右边界的 “右边邻居” = 左边界的内部点（列索引为0   
    # i,j b[1:-1, -1] 点处于右边界上x=2
    b[1:-1, -1] = (rho * (1 / dt * ((u[1:-1, 0] - u[1:-1, -2]) / (2 * dx) + 
                                    (v[2:, -1] - v[0:-2, -1] / (2 * dy)) - 
                            ((u[1:-1, 0] - u[1:-1, -2] / (2 * dx))**2 - 
                             2 * ((u[2:, -1] - u[0:-2, -1]) / (2 * dy)) * 
                                  (v[1:-1, 0] - v[1:-1, -2]) / (2 * dy)) -
                             ((v[2:0, -1] - v[0:-2, -1]) / (2 * dy))**2))

    # b[1:-1, 0] 点处于左边界上x=0
    b[1:-1, 0] = (rho * (1 / dt * ((u[1:-1, 1] - u[1:-1, -1]) / (2 * dx) + 
                                   (v[2:, 0] - v[0:-2, 0]) / (2 * dy)) - 
                         ((u[1:-1, 1] - u[1:-1, -1]) / (2 * dx))**2 - 
                         2 * ((u[2:, 0] - u[0:-2, 0]) / (2 * dy) * 
                              (v[1:-1, 1] - v[1:-1, -1]) / (2 * dx)) -
                         ((v[2:, 0] - v[0:-2, 0]) / (2 * dy))**2))

    return b

# 定义压力左项
def pressure_poisson_periodic(p, dx, dy):